# TP Integrador — Modulación LoRa / FSCM

Materia: Procesamiento de Señales
Grupo 8: Testigos de Vangelista
Integrantes: Castro, Victor José Arturo · Carreras, Leonor Victoria

El trabajo sigue el artículo de Lorenzo Vangelista, "Frequency Shift Chirp
Modulation: The LoRa Modulation". La cadena completa es:

    1. Parámetros del PHY
    2. Bits -> símbolos            (Ec. 1)
    3. Modulación                  (Ec. 2 y 3)
    4. Demodulación                (Ec. 15 y 16)
    5. Trama y preámbulo
    6. Etapa SDR: configuración y emisión
    7. Sincronización temporal por correlación
    8. Cadena completa en simulación
    9. Recepción desde el SDR: buffer de dos tramas
   10. PER sobre el enlace real
   11. Cierre del enlace
   12. Conclusiones

Todo el archivo corre de arriba hacia abajo, sin argumentos de línea de comandos.
Cada sección termina mostrando un resultado: un `print` o una figura.

Nota sobre la tasa de muestreo: el modelo del artículo trabaja a una muestra por
chip, con `T = 1/B`. El Pluto transmite y recibe a `SamplingRate = 1 MSPS`, y el
chirp barre exactamente ese ancho de banda. Por eso acá no hace falta ninguna
etapa de remuestreo: `Fs = B = 1 MHz` y `M = 2**SF` muestras por símbolo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Parámetros del PHY

El artículo define el chirp sobre `M = 2**SF` muestras por símbolo, con un
período de chip `T = 1/B`:

$$c(nT_s + kT) = \frac{1}{\sqrt{2^{SF}}}\,
e^{j2\pi\frac{[(s+k) \bmod 2^{SF}]\cdot k}{2^{SF}}}$$

Elegimos `SF = 8`, que da `M = 256` muestras por símbolo y 256 símbolos
posibles (8 bits por símbolo). La tasa de muestreo del SDR es `1 MSPS`, así que
el chirp barre `1 MHz` y un símbolo dura `M/Fs = 256 µs`.

Ojo con no confundir dos frecuencias distintas:

* `B` es el **ancho de banda del chirp** (1 MHz). Es el `B` del artículo.
* `fc` es la **portadora** (928 MHz). No aparece en las ecuaciones; solo traslada
  el espectro a la banda ISM.

In [ ]:
SF = 8                        # bits por símbolo
SamplingRate = 1_000_000      # tasa del SDR [muestras/s]
B = SamplingRate              # el chirp barre exactamente la tasa de muestreo
M = 2 ** SF                   # muestras por símbolo = símbolos posibles
T = 1 / B                     # período de chip [s]
Ts = M * T                    # duración de un símbolo [s]
k = np.arange(M)              # índice de chip 0 .. M-1

fc = 928e6                    # portadora [Hz]

n_up = 8                      # up-chirps del preámbulo
n_down = 2                    # down-chirps completos
cuarto_down = True            # agregar el último 1/4 de down-chirp
n_simbolos = 10               # símbolos de payload
tramas_en_buffer = 2          # tramas que se guardan en el receptor (2x la enviada)
n_paquetes_per = 30000        # paquetes a recibir para medir el PER

print(f"SF              = {SF}")
print(f"M               = {M} muestras/símbolo, {M} símbolos posibles")
print(f"B (chirp)       = {B/1e6:.3f} MHz")
print(f"fc (portadora)  = {fc/1e6:.0f} MHz")
print(f"T               = {T*1e6:.3f} µs")
print(f"Ts              = {Ts*1e6:.3f} µs")
print(f"Bits por símbolo = {SF}")
print(f"Resolución en frecuencia = {B/M:.1f} Hz")
print(f"Buffer de RX    = {tramas_en_buffer} tramas")
print(f"Paquetes de PER = {n_paquetes_per}")

# 2. Bits -> símbolos (Ecuación 1)

La ecuación (1) arma el símbolo como una suma ponderada de los bits, con el bit
menos significativo primero:

$$s(nT_s) = \sum_{h=0}^{SF-1} w(nT_s)_h \cdot 2^h$$

Es decir, un número binario escrito al revés. Con `SF = 8` hay 256 símbolos.

In [ ]:
# Bloque de bits aleatorio pero determinista: con la misma semilla el resultado
# es siempre el mismo; cambiando la semilla cambian los bits.
semilla_bits = 2024
generador_bits = np.random.default_rng(semilla_bits)
bits_payload = generador_bits.integers(0, 2, size=(n_simbolos, SF))

# Ecuación 1: cada fila de bits (LSB primero) pasa a un símbolo entero.
simbolos_payload = []
for fila in bits_payload:
    fila = np.asarray(fila, dtype=np.int64).ravel()
    pesos = 2 ** np.arange(fila.size)
    simbolos_payload.append(int(np.dot(fila, pesos)))

print(f"Semilla del bloque de bits: {semilla_bits}")
print("Bits del payload (LSB primero):")
for indice in range(n_simbolos):
    print(f"  {bits_payload[indice].tolist()} -> {simbolos_payload[indice]:3d}")

# Ida y vuelta: símbolo -> bits (LSB primero).
verificacion = []
for simbolo in simbolos_payload:
    valor = int(simbolo)
    verificacion.append([(valor >> h) & 1 for h in range(SF)])
print(f"Ida y vuelta exacta: {verificacion == bits_payload.tolist()}")

# 3. Modulación (Ecuaciones 2 y 3)

El chirp base para el símbolo `s` es

$$c_s(k) = \frac{1}{\sqrt{M}} e^{j2\pi\frac{[(s+k) \bmod M]\cdot k}{M}},
\quad k = 0, 1, \dots, M-1$$

El corazón es `exp(j2πk²/M)`: la frecuencia instantánea crece linealmente con
`k`. El desplazamiento `s` cambia el punto de arranque del barrido, y como el
índice va módulo `M`, el chirp da la vuelta cuando se pasa del final. Eso es
exactamente lo que hace ortogonales a los `M` símbolos.

`FSK` aparece como contraste: en lugar de barrer, salta a una frecuencia fija.

El barrido avanza dos ciclos por símbolo, así que la frecuencia recorre el
ancho de banda dos veces. La figura siguiente muestra **una sola vuelta**,
desde `-B/2` hasta `+B/2`: al llegar al máximo la frecuencia corta abruptamente
y vuelve al mínimo.


In [ ]:
def up_chirp(M=M):
    """Chirp ascendente base, energía unitaria."""
    kk = np.arange(M)
    return (1.0 / np.sqrt(M)) * np.exp(1j * 2 * np.pi * (kk ** 2) / M)


def down_chirp(M=M):
    """Chirp descendente: el conjugado del ascendente."""
    return np.conj(up_chirp(M))


def down_chirp_demod(M=M):
    """Réplica usada para dechirpear. Difiere del down-chirp en la escala 1/sqrt(M)."""
    kk = np.arange(M)
    return np.exp(-1j * 2 * np.pi * (kk ** 2) / M)


def modular(simbolos, M=M):
    """Genera la señal FSCM de una secuencia de símbolos (Ec. 2 y 3)."""
    kk = np.arange(M)
    partes = []
    for simbolo in np.asarray(simbolos).ravel():
        if not 0 <= int(simbolo) < M:
            raise ValueError(f"símbolo fuera de rango: {simbolo}")
        fase = 2 * np.pi * (((int(simbolo) + kk) % M) * kk) / M
        partes.append((1 / np.sqrt(M)) * np.exp(1j * fase))
    return np.concatenate(partes)

In [ ]:
# --- Demostración: el último símbolo del payload, en tiempo, frecuencia y
# espectrograma ---
simbolo_demo = int(simbolos_payload[-1])
senal_demo = modular([simbolo_demo])

# El barrido avanza dos ciclos por símbolo, así que la frecuencia instantánea
# recorre el ancho de banda dos veces: sube de -B/2 a +B/2 y salta de vuelta.
# Para el gráfico se toma media vuelta (M/2 muestras): desde el salto hasta que
# vuelve a llegar a +B/2, donde la frecuencia corta abruptamente.
delta = np.diff(np.unwrap(np.angle(senal_demo)))
f_unwrapped = delta / (2 * np.pi)                    # ciclos/muestra
f_fold = (f_unwrapped + 0.5) % 1.0 - 0.5             # plegada a (-0.5, 0.5]
saltos = np.where(np.diff(f_fold) < -0.5)[0] + 1    # cortes +B/2 -> -B/2
inicio_vuelta = int(saltos[0])
fin_vuelta = int(saltos[1]) if len(saltos) > 1 else M
senal_vuelta = senal_demo[inicio_vuelta:fin_vuelta]
f_vuelta = f_fold[inicio_vuelta:fin_vuelta] * SamplingRate / 1e3   # kHz

fig, axes = plt.subplots(3, 1, figsize=(12, 9))
axes[0].plot(np.real(senal_vuelta), label="parte real (I)", linewidth=1)
axes[0].plot(np.imag(senal_vuelta), label="parte imaginaria (Q)", linewidth=1)
axes[0].set_title(
    f"Chirp del símbolo {simbolo_demo}: una vuelta ({len(senal_vuelta)} muestras)")
axes[0].set_xlabel("muestra dentro de la vuelta")
axes[0].set_ylabel("amplitud")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# Frecuencia instantánea: una sola rampa, de la mínima a la máxima.
axes[1].plot(f_vuelta, linewidth=1.6)
axes[1].axhline(+B / 2e3, color="tab:red", linestyle=":", linewidth=1)
axes[1].axhline(-B / 2e3, color="tab:red", linestyle=":", linewidth=1)
axes[1].annotate("corte abrupto", xy=(len(f_vuelta) - 1, f_vuelta[-1]),
                 xytext=(-70, -25), textcoords="offset points",
                 arrowprops=dict(arrowstyle="->", linewidth=1))
axes[1].set_title("Frecuencia instantánea: de -B/2 a +B/2 y ahí termina")
axes[1].set_xlabel("muestra dentro de la vuelta")
axes[1].set_ylabel("frecuencia [kHz]")
axes[1].grid(True, linestyle="--", alpha=0.6)

# Espectrograma del símbolo completo: la frecuencia recorre el ancho de banda
# dos veces por símbolo, así que se ven dos dientes de sierra.
axes[2].specgram(senal_demo, NFFT=64, Fs=SamplingRate, noverlap=48,
                 cmap="viridis")
axes[2].set_title("Espectrograma: la frecuencia barre todo el ancho de banda")
axes[2].set_xlabel("tiempo [s]")
axes[2].set_ylabel("frecuencia [Hz]")
fig.tight_layout()
plt.show()


# Contraste FSCM vs FSK en el espectro. FSK usa un tono fijo por símbolo, sin
# barrido; acá se modula el mismo símbolo que en la figura anterior.
senal_fsk = (1 / np.sqrt(M)) * np.exp(
    1j * 2 * np.pi * simbolo_demo * np.arange(M) / M)
espectro_fscm = np.abs(np.fft.fftshift(np.fft.fft(senal_demo)))
espectro_fsk = np.abs(np.fft.fftshift(np.fft.fft(senal_fsk)))

plt.figure(figsize=(12, 4))
plt.plot(np.fft.fftshift(np.fft.fftfreq(M, d=T)) / 1e3, espectro_fscm,
         label="FSCM (chirp)", linewidth=1.5)
plt.plot(np.fft.fftshift(np.fft.fftfreq(M, d=T)) / 1e3, espectro_fsk,
         label="FSK (tono fijo)", linewidth=1.2, alpha=0.8)
plt.title("FSCM ocupa todo el ancho de banda; FSK concentra la energía")
plt.xlabel("frecuencia [kHz]")
plt.ylabel("magnitud")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()

# 4. Demodulación (Ecuaciones 15 y 16)

El receptor óptimo multiplica la señal por el down-chirp y aplica la FFT. La
multiplicación cancela la rampa cuadrática y deja un tono puro, cuya frecuencia
es el símbolo transmitido:

$$Y[k] = \mathrm{FFT}\{r[n]\cdot e^{-j2\pi n^2/M}\}$$

El índice del pico es el símbolo. Por eso el receptor es una FFT por símbolo: no
hay que probar las 256 formas de onda.

In [ ]:
def demodular(recibido, M=M):
    """Demodula uno o varios símbolos. Devuelve los símbolos estimados."""
    datos = np.asarray(recibido, dtype=np.complex128)
    if datos.ndim == 1:
        if datos.size % M != 0:
            raise ValueError("la señal no es múltiplo de M")
        datos = datos.reshape(-1, M)
    dechirpeado = datos * down_chirp_demod(M)[None, :]
    espectros = np.abs(np.fft.fft(dechirpeado, axis=1))
    return np.argmax(espectros, axis=1)


def espectro_demodulado(muestra, M=M):
    """Espectro tras el dechirp de un único símbolo."""
    return np.abs(np.fft.fft(np.asarray(muestra) * down_chirp_demod(M)))

In [ ]:
# --- Demostración: espectro con el pico marcado ---
espectro = espectro_demodulado(senal_demo)
simbolo_recuperado = int(np.argmax(espectro))

plt.figure(figsize=(12, 4))
plt.plot(np.arange(M), espectro, linewidth=1.4)
plt.axvline(simbolo_recuperado, color="red", linestyle="--",
            label=f"pico en {simbolo_recuperado}")
plt.title(f"Dechirp + FFT: el pico cae en el símbolo transmitido ({simbolo_demo})")
plt.xlabel("bin / símbolo")
plt.ylabel("magnitud")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()

print(f"Símbolo transmitido: {simbolo_demo}")
print(f"Símbolo recuperado:  {simbolo_recuperado}")
print(f"Correcto: {simbolo_recuperado == simbolo_demo}")

# 5. Trama y preámbulo

La trama tiene dos partes:

* **Preámbulo**: 8 up-chirps idénticos, 2 down-chirps completos y un último
  cuarto de down-chirp. El receptor lo usa para encontrar el comienzo. Los
  down-chirps finales rompen la repetición de los up-chirps: sin ellos, la
  correlación tendría un pico por cada copia del up-chirp.
* **Payload**: `n_simbolos` símbolos.

Longitudes, en muestras:

    preámbulo = (n_up + n_down)*M + (M/4 si cuarto_down)
    trama     = preámbulo + n_simbolos*M

Con `SF = 8` y los valores por defecto: preámbulo de 2624 muestras y trama de
5184. Al cambiar `SF` cambian `M` y todas las longitudes derivadas.

In [ ]:
def preambulo(M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    """Construye el preámbulo: up-chirps, down-chirps y 1/4 de down-chirp."""
    partes = [np.tile(up_chirp(M), n_up)]
    if n_down:
        partes.append(np.tile(down_chirp(M), n_down))
    if cuarto:
        partes.append(down_chirp(M)[: M // 4])
    return np.concatenate(partes)


def longitud_preambulo(M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    largo = n_up * M + n_down * M
    if cuarto:
        largo += M // 4
    return largo


def construir_trama(simbolos, M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    """Preámbulo + payload modulado."""
    return np.concatenate([
        preambulo(M, n_up, n_down, cuarto),
        modular(simbolos, M),
    ])


def longitud_trama(M=M, n_simbolos=n_simbolos, n_up=n_up, n_down=n_down,
                   cuarto=cuarto_down):
    return longitud_preambulo(M, n_up, n_down, cuarto) + n_simbolos * M


preambulo_ref = preambulo()
trama = construir_trama(simbolos_payload)

print(f"Preámbulo: {preambulo_ref.size} muestras "
      f"({preambulo_ref.size/SamplingRate*1e3:.3f} ms)")
print(f"Payload:   {n_simbolos*M} muestras")
print(f"Trama:     {trama.size} muestras ({trama.size/SamplingRate*1e3:.3f} ms)")
esperado_preambulo = (n_up + n_down) * M + (M // 4 if cuarto_down else 0)
esperado_trama = esperado_preambulo + n_simbolos * M
print(f"Esperado:  preámbulo {esperado_preambulo}, trama {esperado_trama}")

In [ ]:
# --- Figura: la trama completa, con el límite entre preámbulo y payload ---
plt.figure(figsize=(12, 3.5))
plt.plot(np.real(trama), linewidth=0.7)
plt.axvline(n_up * M, color="orange", linestyle=":",
            label=f"fin de los up-chirps ({n_up * M})")
plt.axvline(longitud_preambulo(), color="red", linestyle="--",
            label=f"fin del preámbulo ({longitud_preambulo()})")
plt.title("Trama completa: preámbulo (up + down) y payload")
plt.xlabel("muestra")
plt.ylabel("Re{ }")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()

In [ ]:
# --- Figura: la trama por segmentos ---
segmentos = [(f"{n_up} up-chirps", 0, n_up * M)]
if n_down:
    segmentos.append((f"{n_down} down", n_up * M, (n_up + n_down) * M))
if cuarto_down:
    segmentos.append(("1/4 down", (n_up + n_down) * M, longitud_preambulo()))
segmentos.append(("payload", longitud_preambulo(), trama.size))

fig, axes = plt.subplots(len(segmentos), 1, figsize=(12, 2.0 * len(segmentos)))
for eje, (nombre, ini, fin) in zip(axes, segmentos):
    eje.plot(np.real(trama[ini:fin]), linewidth=1)
    eje.set_title(f"{nombre}  ({fin - ini} muestras)", fontsize=10)
    eje.set_ylabel("Re{ }")
    eje.grid(True, linestyle="--", alpha=0.5)
axes[-1].set_xlabel("muestra dentro del segmento")
fig.suptitle("Estructura de la trama", fontweight="bold")
fig.tight_layout()
plt.show()

# 6. Etapa SDR (ADALM-PLUTO): configuración y emisión

El camino al laboratorio es: VPN de la cátedra (`200.16.19.5:443`, usuario de
la planilla), JupyterHub (`192.168.1.60`) y, desde ahí, el Pluto asignado. En
la planilla actualizada, el SDR del grupo 8 ("Testigos de Vangelista") es
**192.168.1.34**; por eso `Uri` apunta a esa dirección.

Primero se configura el enlace completo, en el mismo orden que el instructivo
"How To Config SDR": lazo de retorno, frecuencia de muestreo, ancho de banda de
RF, ganancias, portadoras y buffers. Recién después se envía la trama.

Esta sección **no captura ni decodifica**: solo configura y deja el transmisor
emitiendo. La captura y la decodificación están en §9, y la medición del PER en
§10. Por eso el enlace queda abierto hasta el cierre de §11.

Lo importante para la recepción es que el transmisor emite desde un buffer
cíclico (`sdr.tx_cyclic_buffer = True` **antes** de transmitir) y que el
receptor pide **dos tramas**:

    sdr.rx_buffer_size = 2 * len(trama)

Con dos tramas alcanza para garantizar al menos una trama completa, arranque
donde arranque la captura (ver §9).

No hay `try/except`: si no está el Pluto o no está instalado `pyadi-iio`, la
ejecución se corta en esta sección.

In [ ]:
# --- Configuración del enlace (planilla de grupos actualizada) ---
Uri = "ip:192.168.1.34"        # Pluto asignado al grupo 8
Loopback = 0                   # 0 = deshabilitado, 1 = digital, 2 = retransmisor
TxLOFreq = int(fc)             # portadora de TX [Hz]
RxLOFreq = TxLOFreq            # misma portadora en RX
TxAtten = -60                  # atenuación de TX, de -89 a 0 dB; se arranca bajo
TxRfBw = 2_000_000             # filtro analógico de TX [Hz]
RxRfBw = TxRfBw                # filtro analógico de RX [Hz]
GainControlModes = "slow_attack"
TxCyclicBuffer = True          # el Pluto repite la trama sin pausa
RxBufferSize = tramas_en_buffer * longitud_trama()   # 2 tramas

# El enlace se deja abierto para capturar en §9 y medir el PER en §10;
# el cierre va en §11.
import adi

sdr = adi.Pluto(Uri)

# 1) Lazo de retorno
sdr.loopback = Loopback

# 2) Frecuencia de muestreo (TX y RX)
sdr.sample_rate = int(SamplingRate)

# 3) Ancho de banda de los filtros analógicos de RF
sdr.tx_rf_bandwidth = int(TxRfBw)
sdr.rx_rf_bandwidth = int(RxRfBw)

# 4) Ganancia: atenuación de TX; RX con el AGC (slow_attack)
sdr.tx_hardwaregain_chan0 = int(TxAtten)
sdr.gain_control_mode_chan0 = GainControlModes

# 5) Portadoras de TX y RX
sdr.tx_lo = int(TxLOFreq)
sdr.rx_lo = int(RxLOFreq)

# 6) Buffers: RX de dos tramas y TX cíclico antes de transmitir
sdr.rx_buffer_size = int(RxBufferSize)
sdr.tx_cyclic_buffer = bool(TxCyclicBuffer)

print(f"Pluto conectado en {Uri}")
print(f"  Fs = {SamplingRate/1e6:.3f} MSPS, portadora = {TxLOFreq/1e6:.0f} MHz")
print(f"  trama de TX = {trama.size} muestras")
print(f"  buffer de RX = {RxBufferSize} muestras ({tramas_en_buffer} tramas)")
print("  enlace abierto: la captura va en §9 y el PER en §10")

In [ ]:
# --- Emisión de la trama: el buffer cíclico la repite sin pausa ---
sdr.tx(trama * (2 ** 15 - 1))
print(f"TX encendido: {trama.size} muestras escaladas a 16 bits.")
print("El Pluto reinyecta la trama solo; el enlace queda abierto para §9 y §10.")

# 7. Sincronización temporal por correlación

Con la trama en el aire, el receptor tiene que responder una pregunta: ¿en qué
muestra empieza? La respuesta es la correlación entre la captura y el preámbulo
completo: donde el preámbulo cae alineado, los chirps coinciden y la suma da un
pico; desplazado, los chirps no coinciden y las contribuciones se cancelan.

La correlación se calcula por FFT, porque en el dominio del tiempo cuesta
`N·L` multiplicaciones y por FFT cuesta del orden de `N·log(N)`:

$$r[k] = \mathrm{IFFT}\left(\mathrm{FFT}(r)\cdot\overline{\mathrm{FFT}(p)}\right)$$

En el lazo local el transmisor y el receptor comparten el oscilador, así que no
hay corrimiento de frecuencia que deforme la correlación: alcanza con buscar el
máximo.

In [ ]:
def correlacion_fft(recibido, patron):
    """|correlación| entre señal y patrón, calculada con FFT."""
    recibido = np.asarray(recibido, dtype=np.complex128)
    patron = np.asarray(patron, dtype=np.complex128)
    if recibido.size < patron.size:
        raise ValueError("la señal es más corta que el patrón")
    n = recibido.size + patron.size - 1
    nfft = 1 << (n - 1).bit_length()
    producto = np.fft.fft(recibido, nfft) * np.conj(np.fft.fft(patron, nfft))
    # `ifft` ya trae el factor 1/nfft: el resultado es la suma, no el promedio.
    completo = np.fft.ifft(producto)
    return np.abs(completo[: recibido.size - patron.size + 1])


def sincronizar(recibido, M=M):
    """Ubica el inicio de la trama correlacionando con el preámbulo completo.

    Devuelve el `inicio` y una `metrica` entre 0 y 1 que mide cuánto se parece
    el tramo encontrado al preámbulo (1 = idéntico salvo escala).
    """
    recibido = np.asarray(recibido, dtype=np.complex128)
    patron = preambulo(M)
    correlacion = correlacion_fft(recibido, patron)
    inicio = int(np.argmax(correlacion))
    energia = float(np.sum(np.abs(recibido[inicio: inicio + patron.size]) ** 2))
    metrica = float(correlacion[inicio] / (np.linalg.norm(patron) * np.sqrt(energia)))
    return {"inicio": inicio, "metrica": metrica, "correlacion": correlacion}

In [ ]:
# --- Demostración: trama con offset y ruido, se detecta el inicio ---
rng = np.random.default_rng(2024)
desfase_demo = int(rng.integers(int(0.3 * trama.size), int(3.0 * trama.size)))
ruido_demo = (rng.normal(size=trama.size) + 1j * rng.normal(size=trama.size)) * 0.02

captura_demo = np.zeros(desfase_demo + trama.size, dtype=np.complex128)
captura_demo[desfase_demo:] = trama + ruido_demo

info_demo = sincronizar(captura_demo)

plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1)
plt.plot(np.abs(captura_demo), linewidth=0.6, color="tab:green")
plt.axvline(desfase_demo, color="green", linestyle="--", label="inicio real")
plt.title("Captura: la trama arranca después de un tramo de silencio")
plt.xlabel("muestra")
plt.ylabel("|captura|")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(info_demo["correlacion"], linewidth=1)
plt.axvline(desfase_demo, color="green", linestyle="--", label="inicio real")
plt.axvline(info_demo["inicio"], color="red", linestyle="--",
            label=f"inicio detectado ({info_demo['inicio']})")
plt.title("Correlación con el preámbulo completo: el máximo marca el inicio")
plt.xlabel("muestra")
plt.ylabel("|correlación|")
plt.xlim(0, min(captura_demo.size, info_demo["inicio"] + 4 * M))
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Inicio real:      {desfase_demo}")
print(f"Inicio detectado: {info_demo['inicio']}  (error {info_demo['inicio'] - desfase_demo:+d})")
print(f"Métrica:          {info_demo['metrica']:.3f}")

# 8. Cadena completa en simulación

`extraer_payload` recorta la ventana del payload a partir del inicio detectado y
`demodular_trama` encadena sincronización, extracción y demodulación de una trama
entera. `recepcion_simulada` arma la captura de `tramas_en_buffer` tramas que usa
la sección 9 para verificar el tratamiento circular.

In [ ]:
def extraer_payload(recibido, inicio, M=M, n_simbolos=n_simbolos):
    """Devuelve la matriz `(n_simbolos, M)` del payload, después del preámbulo."""
    comienzo = int(inicio) + longitud_preambulo(M)
    fin = comienzo + n_simbolos * M
    recibido = np.asarray(recibido, dtype=np.complex128)
    if comienzo < 0 or fin > recibido.size:
        raise ValueError("la ventana del payload se sale de la señal")
    return recibido[comienzo:fin].reshape(n_simbolos, M)


def demodular_trama(recibido, M=M, n_simbolos=n_simbolos):
    """Sincroniza, extrae y demodula el payload de una trama."""
    info = sincronizar(recibido, M)
    payload = extraer_payload(recibido, info["inicio"], M, n_simbolos)
    return demodular(payload, M), info


def recepcion_simulada(simbolos, offset, snr_db=None, semilla=0):
    """Genera una captura como la devolvería el SDR: `tramas_en_buffer` tramas + ruido."""
    señal = construir_trama(simbolos)
    largo = señal.size
    buffer = np.zeros(tramas_en_buffer * largo, dtype=np.complex128)
    # Se escribe de forma circular: si la trama no entra, sigue por el principio.
    buffer[(np.arange(largo) + offset) % buffer.size] += señal

    if snr_db is not None:
        generador = np.random.default_rng(semilla)
        potencia = np.mean(np.abs(buffer) ** 2)
        ruido = (generador.normal(size=buffer.size)
                 + 1j * generador.normal(size=buffer.size))
        buffer += ruido * np.sqrt(potencia / (10 ** (snr_db / 10)) / 2)
    return buffer

# 9. Recepción desde el SDR: buffer de dos tramas

El transmisor emite la trama en un **buffer cíclico**: la repite sin pausa. El
receptor, cuando dispara `sdr.rx()`, no sabe en qué punto del ciclo está la
trama. Por eso el buffer de recepción se dimensiona con **dos tramas**
(`RxBufferSize = 2 * len(trama)`): una ventana de dos períodos contiene siempre
al menos una trama completa, sin importar dónde arranque.

Pero guardar dos tramas no alcanza: la trama puede empezar cerca del final y
terminar al principio del buffer. Si el receptor lee el payload con
`inicio + i*M` y se queda sin muestras, pierde los últimos símbolos.

La solución es tratar el buffer como **circular**. Se extiende por los dos
lados —anteponiendo el final y agregando el comienzo— de modo que la trama
aparezca contigua en alguna copia, se detecta el preámbulo sobre esa copia y se
lee el payload completo, dé la vuelta o no.

In [ ]:
def extender_circular(buffer, largo):
    """Extiende un buffer circular por ambos lados.

    Devuelve `(extendido, pad)`, donde `pad` son las muestras antepuestas. Un
    inicio detectado en `extendido` se lleva al buffer con
    `(inicio - pad) % len(buffer)`.
    """
    buffer = np.asarray(buffer, dtype=np.complex128)
    if buffer.ndim != 1:
        raise ValueError("el buffer debe ser un vector complejo 1-D")
    if buffer.size == 0:
        raise ValueError("el buffer está vacío")
    tramo = min(int(largo), buffer.size)
    extendido = np.concatenate([buffer[-tramo:], buffer, buffer[:tramo]])
    return extendido, tramo


def tamaño_buffer_rx(M=M, n_simbolos=n_simbolos, tramas=tramas_en_buffer):
    """Muestras que hay que pedirle al SDR: una trama por cada copia."""
    return tramas * longitud_trama(M, n_simbolos)


def demodular_buffer(buffer, M=M, n_simbolos=n_simbolos):
    """Decodifica una trama contenida en un buffer circular del SDR.

    Acepta la trama esté donde esté y dé la vuelta o no. Devuelve
    `(simbolos, info)`, con `info['inicio']` reducido al buffer original.
    """
    buffer = np.asarray(buffer, dtype=np.complex128)
    if buffer.ndim != 1:
        raise ValueError("el buffer debe ser un vector complejo 1-D")

    una_trama = longitud_trama(M, n_simbolos)
    if buffer.size < una_trama:
        raise ValueError("el buffer no alcanza para una trama")
    if buffer.size < 2 * una_trama:
        # Con una sola trama el buffer es periódico y el inicio queda ambiguo
        # módulo la trama; completar con ceros rompe la simetría.
        faltan = 2 * una_trama - buffer.size
        buffer = np.concatenate([buffer, np.zeros(faltan, dtype=np.complex128)])

    extendido, pad = extender_circular(buffer, una_trama)
    info = sincronizar(extendido, M)

    # Se lee la copia central, la que cae dentro del buffer, para que la ventana
    # del payload siempre entre completa.
    inicio = (info["inicio"] - pad) % buffer.size
    payload = extraer_payload(extendido, inicio + pad, M, n_simbolos)
    info["inicio"] = int(inicio)
    info["pad"] = int(pad)
    return demodular(payload, M), info


print(f"Buffer de recepción: {tramas_en_buffer} tramas = "
      f"{tamaño_buffer_rx()} muestras "
      f"({tamaño_buffer_rx()/SamplingRate*1e3:.3f} ms)")

print("\nPrueba A: la trama arranca justo en el borde del buffer")
buffer_a = recepcion_simulada(simbolos_payload, 0, snr_db=15.0, semilla=1)
simbolos_a, info_a = demodular_buffer(buffer_a)
print(f"  inicio detectado: {info_a['inicio']}")
print(f"  recibidos: {simbolos_a.tolist()}")
print(f"  exacto: {simbolos_a.tolist() == simbolos_payload}")

print("\nPrueba B: la trama da la vuelta por el final del buffer")
offset_b = tamaño_buffer_rx() - 300
buffer_b = recepcion_simulada(simbolos_payload, offset_b, snr_db=15.0, semilla=2)
simbolos_b, info_b = demodular_buffer(buffer_b)
print(f"  offset real: {offset_b} de {buffer_b.size}")
print(f"  inicio detectado: {info_b['inicio']}")
print(f"  recibidos: {simbolos_b.tolist()}")
print(f"  exacto: {simbolos_b.tolist() == simbolos_payload}")

print("\nPrueba C: la trama arranca en la segunda mitad del buffer")
buffer_c = recepcion_simulada(simbolos_payload, tamaño_buffer_rx() // 2 + 123,
                              snr_db=10.0, semilla=3)
simbolos_c, info_c = demodular_buffer(buffer_c)
print(f"  inicio detectado: {info_c['inicio']}")
print(f"  recibidos: {simbolos_c.tolist()}")
print(f"  exacto: {simbolos_c.tolist() == simbolos_payload}")

In [ ]:
# --- Figura: la captura circular y el inicio detectado ---
plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1)
plt.plot(np.real(buffer_b), linewidth=0.8)
plt.axvline(info_b["inicio"], color="red", linestyle="--",
            label=f"inicio detectado ({info_b['inicio']})")
plt.axvline(info_b["inicio"] + longitud_trama(), color="orange", linestyle=":",
            label="fin de la trama")
plt.title("Captura de dos tramas: la trama cruza el final del buffer")
plt.xlabel("muestra del buffer")
plt.ylabel("Re{ }")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(2, 1, 2)
# Igual que en `demodular_buffer`: se extiende el buffer y se lee el primer
# símbolo del payload desde la copia central.
buffer_b_extendido, _ = extender_circular(buffer_b, longitud_trama())
espectro_b = espectro_demodulado(
    buffer_b_extendido[
        info_b["inicio"] + longitud_preambulo():
        info_b["inicio"] + longitud_preambulo() + M
    ]
)
plt.plot(np.arange(M), espectro_b, linewidth=1.2)
plt.axvline(int(np.argmax(espectro_b)), color="red", linestyle="--",
            label=f"símbolo {int(np.argmax(espectro_b))}")
plt.title("FFT tras dechirp del primer símbolo del payload")
plt.xlabel("bin / símbolo")
plt.ylabel("magnitud")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Captura real del enlace y decodificación ---
captura = sdr.rx()

if captura.size < tramas_en_buffer * longitud_trama():
    print(f"Advertencia: se pidieron {tramas_en_buffer * longitud_trama()} "
          f"muestras y llegaron {captura.size}. Revisar rx_buffer_size.")

simbolos_rx, info_rx = demodular_buffer(captura)
print(f"Inicio detectado: {info_rx['inicio']}")
print(f"Métrica:          {info_rx['metrica']:.3f}")
print(f"Símbolos:         {simbolos_rx.tolist()}")
print(f"¿Trama correcta?  {simbolos_rx.tolist() == simbolos_payload}")

# --- Figura: la señal recibida por el SDR (dos tramas) ---
plt.figure(figsize=(12, 5))

plt.subplot(2, 1, 1)
plt.plot(np.real(captura), linewidth=0.6, label="parte real")
plt.plot(np.imag(captura), linewidth=0.6, alpha=0.7, label="parte imaginaria")
plt.title("Señal recibida por el SDR (dos tramas)")
plt.xlabel("muestra")
plt.ylabel("amplitud")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")

plt.subplot(2, 1, 2)
plt.plot(np.abs(captura), linewidth=0.6, color="tab:green")
plt.title("Envolvente de la señal recibida (el rizado es el ruido del canal)")
plt.xlabel("muestra")
plt.ylabel("|captura|")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# 10. PER (Packet Error Rate) sobre el enlace real

El transmisor cicla siempre la **misma** trama, así que cada `sdr.rx()` es una
observación nueva del mismo paquete. Como el receptor conoce los símbolos
transmitidos (`simbolos_payload`), alcanza con decodificar y comparar: un
paquete es erróneo si al menos uno de sus `n_simbolos` símbolos no coincide.

Se acumulan `n_paquetes_per = 30000` paquetes y se informa

    PER = paquetes_erroneos / paquetes_recibidos

Con `N = 30000`, un solo error da `3.3e-5` y el techo con cero errores es
`1e-4`, que es lo que hace medible un PER bajo.

Cada captura aporta un paquete (la copia central que decodifica
`demodular_buffer`), así que hacen falta 30 000 capturas de 10.368 ms: unos
5 minutos de señal más el procesamiento. Los contadores se acumulan entre
corridas de la celda, de modo que se puede cortar y retomar sin perder lo
medido; para reiniciar, ejecutar las tres asignaciones a cero que están
comentadas al principio del bloque. Con `Loopback = 1` el PER medido debe dar 0:
es el control de la cadena completa.

In [ ]:
# --- Contadores del PER: se acumulan entre corridas de esta celda ---
if "paquetes_recibidos" not in globals():
    paquetes_recibidos = 0
    paquetes_erroneos = 0
    simbolos_erroneos = 0
    historial_per = []

# Para empezar de cero sin cerrar el enlace:
# paquetes_recibidos = 0; paquetes_erroneos = 0; simbolos_erroneos = 0
# historial_per = []

objetivo_per = max(0, n_paquetes_per - paquetes_recibidos)
progreso_per = max(1, objetivo_per // 10)
simbolos_tx = np.asarray(simbolos_payload)
if paquetes_recibidos:
    historial_per.append((paquetes_recibidos,
                          paquetes_erroneos / paquetes_recibidos))

for indice_per in range(objetivo_per):
    captura_per = sdr.rx()
    simbolos_per, _ = demodular_buffer(captura_per)
    errores_este = int(np.sum(simbolos_per != simbolos_tx))
    simbolos_erroneos += errores_este
    paquetes_recibidos += 1
    if errores_este:
        paquetes_erroneos += 1
    if (indice_per + 1) % progreso_per == 0:
        per_parcial = paquetes_erroneos / paquetes_recibidos
        historial_per.append((paquetes_recibidos, per_parcial))
        print(f"  {paquetes_recibidos:6d} paquetes  "
              f"paquetes erróneos={paquetes_erroneos:4d}  PER={per_parcial:.2e}")

if paquetes_recibidos:
    PER = paquetes_erroneos / paquetes_recibidos
    print(f"\nPaquetes recibidos: {paquetes_recibidos} de {n_paquetes_per}")
    print(f"Paquetes erróneos:  {paquetes_erroneos}")
    print(f"Símbolos erróneos:  {simbolos_erroneos} de "
          f"{paquetes_recibidos * n_simbolos}")
    print(f"PER = {PER:.3e}")
else:
    print("No hay paquetes medidos todavía: correr esta celda con el enlace abierto.")

# --- Figura: PER acumulado contra paquetes recibidos ---
if len(historial_per) > 1:
    paquetes_hist, per_hist = zip(*historial_per)
    plt.figure(figsize=(12, 4))
    plt.plot(paquetes_hist, per_hist, marker="o", linewidth=1.4)
    plt.title("PER acumulado sobre el enlace real")
    plt.xlabel("paquetes recibidos")
    plt.ylabel("PER")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

# 11. Cierre del enlace

Último paso obligatorio: si no se corre, el Pluto queda emitiendo. Va al final
del archivo, después de capturar (§9) y de medir el PER (§10), porque mientras
el enlace esté abierto todavía se pueden sumar paquetes a la medición.

El Pluto no tiene un `tx_off`: la única forma de callarlo es destruir el buffer
de transmisión, bajarle la potencia y sacarlo de la banda de trabajo, igual que
en el cierre del HowTo.

In [ ]:
sdr.tx_destroy_buffer()
sdr.tx_hardwaregain_chan0 = -89
sdr.tx_lo = int(2400e6)
sdr.rx_lo = int(950e6)
sdr.tx(np.zeros(2048))
print("Enlace cerrado.")


# 12. Conclusiones

* La modulación FSCM se describe con una sola fórmula (Ec. 3): un chirp con
  desplazamiento inicial. La ortogonalidad de los `M` símbolos sale de la
  estructura del índice módulo `M`.
* El receptor óptimo es dechirp + FFT + `argmax`: una FFT por símbolo, no una
  correlación contra las `M` formas de onda.
* La trama necesita un preámbulo porque el receptor no sabe cuándo empieza. Los
  up-chirps y los down-chirps finales forman una firma única: la correlación con
  el preámbulo completo tiene un solo pico, y ese pico marca el comienzo.
* En el lazo local el transmisor y el receptor comparten el oscilador, así que no
  hay corrimiento de frecuencia que corregir: la correlación se calcula directo
  sobre la captura.
* En el SDR, el buffer de recepción se dimensiona con dos tramas y se trata como
  circular. Alcanza para garantizar una trama completa y, sin el tratamiento
  circular, una trama que empieza cerca del final se decodifica incompleta, que
  era el síntoma del script de laboratorio.
* El PER se mide sobre el enlace real repitiendo la misma trama: el receptor
  compara lo decodificado con los símbolos transmitidos y acumula paquetes
  erróneos. No hace falta simular Monte Carlo para tener una métrica del enlace.

Fuera de alcance por decisión del grupo: canal AWGN/selectivo simulado, curvas
BER/PER Monte Carlo y retransmisión. El ruido y el canal los aporta el enlace
real entre placas; el PER que se informa es el medido en §10, no una simulación.